# Rung 2 — Grokking Modular Addition (full P=113 run)

Runs the canonical Nanda et al. (ICLR 2023) grokking setup on a free Colab GPU.
On CPU this takes ~5.5h; on a T4 it takes a few minutes per seed.

**Before running:** Runtime -> Change runtime type -> GPU.

This notebook does two runs:
1. **One plotted run** (default seed) -- produces the grokking curve, Fourier
   weights, ablation, and progress-measures figures, and saves the trained
   checkpoint for downstream rungs (Rung 5's SAE, in particular).
2. **A 3-seed run** (`--seeds 0,1,2`) -- produces `results/exp2_grokking.json`,
   a provenance-carrying manifest with the generalization epoch and Fourier
   sparsity reported as mean +/- std across seeds, not a single lucky (or
   unlucky) run. See `06_production_ai/notes/multi-seed-experiment-design.md`.

**Seed budget, decided before running, not after seeing results:** 3 seeds x
5000 epochs at P=113. If none of them grok, that is a genuine negative
result to record -- not a cue to restart with different hyperparameters
until something works (`07_capstone/research-plan.md`'s own fallback clause:
promote Rung 1, induction heads, to the headline).

In [ ]:
BRANCH = "dev"  # change if you're running from a different branch
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
!pip install -q uv
!uv sync --frozen

In [ ]:
# Run 1 of 2: the plotted run. Canonical config -- P=113, 5000 epochs,
# weight_decay=1.0 -- is already every argparse default in exp2_grokking.py,
# so no flags are needed beyond --save-model to keep the checkpoint for
# downstream rungs (Rung 5's real-activation SAE, in particular).
!uv run python -m src.experiments.exp2_grokking --save-model

In [ ]:
# Run 2 of 2: the 3-seed manifest run. Skips plotting/model-saving by design
# (see exp2_grokking.py's --seeds help text) -- this run is about the number
# and its spread, not a second copy of the figures from Run 1.
!uv run python -m src.experiments.exp2_grokking --seeds 0,1,2

In [ ]:
# Zip the figures, the trained checkpoint, and the multi-seed manifest
# together -- the manifest is what lets portfolio/RESULTS.md cite a real
# <!-- manifest: results/exp2_grokking.json --> tag (see
# src/results.py / make verify-claims) instead of a number copied out of
# console scrollback.
!zip -r grokking_results.zip figures/exp2_* results/exp2_grokking.json

from google.colab import files
files.download("grokking_results.zip")

## After downloading

1. Unzip into the local clone: `figures/` for the plots and checkpoint,
   `results/` for the manifest.
2. Read `results/exp2_grokking.json`'s `aggregate` block: `final_val_acc`
   (target: >0.9 mean), `generalization_epoch` (target: well under the 5000-
   epoch budget, with its spread reported -- grokking's generalization epoch
   is known to vary across seeds), `k_99_percent` (target: well under 113,
   e.g. ~10-20 -- the signature sparse Fourier algorithm).
3. Update `portfolio/RESULTS.md` Rung 2's table from the manifest, and add
   `<!-- manifest: results/exp2_grokking.json -->` next to it so
   `make verify-claims` can check it.
4. **If the mean `final_val_acc` across all 3 seeds is still near 1/P (chance)
   after this run, this is a real negative result, not a compute problem
   to solve by re-running.** Record it honestly in the Honesty Ledger and
   invoke the fallback: promote Rung 1 (induction heads) to the headline
   per `07_capstone/research-plan.md`'s flagship strategy. Do not quietly
   retry with different hyperparameters until one seed happens to work --
   that is p-hacking the flagship result.